In [2]:
# Imports + connexion au drive

import pandas as pd

lien = "https://datasets.imdbws.com/title.ratings.tsv.gz"

ratings = pd.read_csv(lien, sep="\t", na_values="\\N")


In [3]:
# Chargement du fichier TMDb The Movie Database

tmdb = pd.read_csv("../data/TMDB_movie_dataset_v11.csv")

#films.to_csv("/content/drive/MyDrive/films_filtres.csv", index=False)

In [5]:
# Fichier IMDb
# title basics : Une ligne par titre : type, titre, année, durée,genres

import pandas as pd

lien = "https://datasets.imdbws.com/title.basics.tsv.gz"

colonnes = ["tconst", "titleType", "primaryTitle", "isAdult", "startYear", "runtimeMinutes", "genres"]

basics = pd.read_csv(lien, sep="\t", na_values="\\N", usecols=colonnes, low_memory=False)

basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

In [6]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2241
1,tt0000002,5.4,330
2,tt0000003,6.4,2412
3,tt0000004,5.0,204
4,tt0000005,6.2,3117


In [8]:
print(basics.shape)
basics.head()

(12979994, 7)


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,0,1894.0,1.0,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,0,1892.0,5.0,"Animation,Short"
2,tt0000003,short,Poor Pierrot,0,1892.0,5.0,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,0,1892.0,12.0,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,0,1893.0,1.0,Short


In [9]:
df = pd.merge(basics, ratings, on="tconst", how="inner")

In [10]:
df.head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000001,short,Carmencita,0,1894.0,1.0,"Documentary,Short",5.7,2241
1,tt0000002,short,Le clown et ses chiens,0,1892.0,5.0,"Animation,Short",5.4,330
2,tt0000003,short,Poor Pierrot,0,1892.0,5.0,"Animation,Comedy,Romance",6.4,2412
3,tt0000004,short,Un bon bock,0,1892.0,12.0,"Animation,Short",5.0,204
4,tt0000005,short,Blacksmith Scene,0,1893.0,1.0,Short,6.2,3117


In [11]:
df.info

<bound method DataFrame.info of             tconst  titleType               primaryTitle  isAdult  startYear  \
0        tt0000001      short                 Carmencita        0     1894.0   
1        tt0000002      short     Le clown et ses chiens        0     1892.0   
2        tt0000003      short               Poor Pierrot        0     1892.0   
3        tt0000004      short                Un bon bock        0     1892.0   
4        tt0000005      short           Blacksmith Scene        0     1893.0   
...            ...        ...                        ...      ...        ...   
1734003  tt9916846  tvEpisode              Episode #3.18        0     2009.0   
1734004  tt9916848  tvEpisode              Episode #3.17        0     2009.0   
1734005  tt9916850  tvEpisode              Episode #3.19        0     2010.0   
1734006  tt9916852  tvEpisode              Episode #3.20        0     2010.0   
1734007  tt9916880  tvEpisode  Horrid Henry Knows It All        0     2014.0   

       

In [28]:
print(f"Point de départ : {df.shape[0]} lignes.")

films = df[df["titleType"] == "movie"]
print(f"Filtré sur 'titleType' = 'Movie' : {films.shape[0]} lignes.")

films = films[films["numVotes"] >= 5000]
print(f"Filtré sur 'numVotes' >= 5000 votes : {films.shape[0]} lignes.")

films = films[films["averageRating"] >= 7.5]
print(f"Filtré sur 'averageRating' >= 7.5 : {films.shape[0]} lignes.")

films = films[films["isAdult"] == 0]
print(f"Filtré sur 'isAdult' : {films.shape[0]} lignes.")

films.head()

Point de départ : 1734008 lignes.
Filtré sur 'titleType' = 'Movie' : 352019 lignes.
Filtré sur 'numVotes' >= 5000 votes : 19291 lignes.
Filtré sur 'averageRating' >= 7.5 : 3323 lignes.
Filtré sur 'isAdult' : 3323 lignes.


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes
2485,tt0006864,movie,Intolerance,0,1916.0,210.0,"Drama,History",7.6,18277
3398,tt0010323,movie,The Cabinet of Dr. Caligari,0,1920.0,67.0,"Horror,Mystery,Thriller",8.0,77573
3967,tt0012349,movie,The Kid,0,1921.0,68.0,"Comedy,Drama,Family",8.2,147895
3972,tt0012364,movie,The Phantom Carriage,0,1921.0,107.0,"Drama,Fantasy,Horror",8.0,16201
4010,tt0012494,movie,Destiny,0,1921.0,114.0,"Drama,Fantasy,Horror",7.6,7525


In [29]:
# Sauvergarde du DataFrame filtré en .csv
films.to_csv("../films.csv", index=False)

In [31]:
mes_films = set(films["tconst"]) 
morceaux_gardes = [] 
lien = "https://datasets.imdbws.com/title.akas.tsv.gz" 

for morceau in pd.read_csv(lien, sep="\t", na_values="\\N",
                           usecols=["titleId", "title", "region"],
                           chunksize=2_000_000):
    est_fr = morceau["region"] == "FR"    
    est_a_moi = morceau["titleId"].isin(mes_films)    
    garde = morceau[est_fr & est_a_moi]    
    morceaux_gardes.append(garde) 
    
titres_fr = pd.concat(morceaux_gardes).drop_duplicates(subset="titleId")

In [32]:
# Fusion de notre liste de films filtrée avec la base akas pour intégrer les titres français.
films = pd.merge(films, titres_fr, 
                    left_on="tconst",
                    right_on="titleId",
                      how="left")

print(f"{films.shape[0]} lignes et {films.shape[1]} colonnes.")
films.head()

3323 lignes et 12 colonnes.


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,titleId,title,region
0,tt0006864,movie,Intolerance,0,1916.0,210.0,"Drama,History",7.6,18277,tt0006864,Intolérance,FR
1,tt0010323,movie,The Cabinet of Dr. Caligari,0,1920.0,67.0,"Horror,Mystery,Thriller",8.0,77573,tt0010323,Le Cabinet du docteur Caligari,FR
2,tt0012349,movie,The Kid,0,1921.0,68.0,"Comedy,Drama,Family",8.2,147895,tt0012349,Le Kid,FR
3,tt0012364,movie,The Phantom Carriage,0,1921.0,107.0,"Drama,Fantasy,Horror",8.0,16201,tt0012364,La charrette fantôme,FR
4,tt0012494,movie,Destiny,0,1921.0,114.0,"Drama,Fantasy,Horror",7.6,7525,tt0012494,Les Trois Lumières,FR


In [33]:
# Ajout des titres "primaryTitle" dans les titres vides de la colonne "title"
films["title"] = films["title"].fillna(films["primaryTitle"])

films["title"].isna().sum()             # ok plus de vide dans "title"

np.int64(0)

In [34]:
# Suppression des colonnes inutiles = "region" ne contient que FR
print(films["region"].value_counts())       # FR = 2704 lignes
print(films.shape)
films = films.drop(columns="region")
print(films.shape)                          # ok colonne supprimée

region
FR    2705
Name: count, dtype: int64
(3323, 12)
(3323, 11)


In [35]:
# Suppression des colonnes inutiles = "titleId" doublon avec colonne "tconst"
films = films.drop(columns="titleId")
print(films.shape)                          # ok colonne supprimée

(3323, 10)


In [36]:
# Vérification de doublons 
films.duplicated().sum()

np.int64(0)

In [37]:
# Sauvegarde après nettoyage
films.to_csv("../films.csv", index=False)